In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
import os

In [2]:
# Get the directory of the current notebook (if it's in the same folder as the CSVs)
project_dir = os.getcwd()
# Now construct the path reliably
normal_csv_path = os.path.join(project_dir, '../data/normal2.csv')
slowed_csv_path = os.path.join(project_dir, '../data/slowed2.csv')

In [3]:
def prepare_data(file_path, label_val, window_size=128):
    df = pd.read_csv(file_path, header=None, names=['raw', 'baseline', 'delta', 'current'])
    data = df['current'].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    data = scaler.fit_transform(data)
    
    windows, labels = [], []
    for i in range(len(data) - window_size):
        windows.append(data[i:i+window_size].T) # Shape [1, 128]
        labels.append(label_val)
    return torch.tensor(np.array(windows), dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

In [4]:
# Load data
X_n, y_n = prepare_data(normal_csv_path, 0)
X_s, y_s = prepare_data(slowed_csv_path, 1)
X = torch.cat([X_n, X_s], dim=0)
y = torch.cat([y_n, y_s], dim=0)

loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

In [5]:
class MotorFaultCNN(nn.Module):
    def __init__(self):
        super(MotorFaultCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        # 128 -> (128-3+1)=126 -> pool(2)=63
        self.fc = nn.Linear(16 * 63, 2) 
        
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [6]:
model = MotorFaultCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
print("Starting training...")
for epoch in range(10): # 10 epochs
    total_loss = 0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

Starting training...
Epoch 1, Loss: 0.6469
Epoch 2, Loss: 0.4965
Epoch 3, Loss: 0.4009
Epoch 4, Loss: 0.3599
Epoch 5, Loss: 0.3229
Epoch 6, Loss: 0.2876
Epoch 7, Loss: 0.2585
Epoch 8, Loss: 0.2321
Epoch 9, Loss: 0.2074
Epoch 10, Loss: 0.1890


In [8]:
# Save the model
torch.save(model.state_dict(), "../models/motor_model.pth")
print("Model saved as motor_model.pth")

Model saved as motor_model.pth


In [9]:
# inference.py
import torch
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Load the model
model = MotorFaultCNN() # Ensure this class is defined exactly as in your training script
model.load_state_dict(torch.load("../models/motor_model.pth"))
model.eval()

# 2. Simulate or capture live data
# This is a dummy example; replace with your actual sensor reading loop
def classify_live_data(live_window):
    # live_window should be a list/array of 128 samples
    data = np.array(live_window).reshape(1, 1, 128)
    data = torch.tensor(data, dtype=torch.float32)
    
    with torch.no_grad():
        output = model(data)
        prediction = torch.argmax(output, dim=1).item()
        
    return "Normal" if prediction == 0 else "Slowed"

# Test it!
print(f"Prediction: {classify_live_data(your_live_128_sample_window)}")

NameError: name 'your_live_128_sample_window' is not defined